In [ ]:
# ============================================================
# CONFIG — imports, chemins, hyperparamètres
# ============================================================

import logging
import os
import warnings
warnings.filterwarnings("ignore")

# ── THREADING ─────────────────────────────────────────────────────────────────
os.environ["OPENBLAS_NUM_THREADS"]   = "128"
os.environ["MKL_NUM_THREADS"]        = "128"
os.environ["OMP_NUM_THREADS"]        = "128"
os.environ["NUMEXPR_NUM_THREADS"]    = "128"
os.environ["VECLIB_MAXIMUM_THREADS"] = "128"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import umap
import hdbscan
torch.set_num_threads(128)

from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from transformers import AutoModel, AutoTokenizer
from wordcloud import WordCloud

# ── LOGS ──────────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")
log = logging.getLogger(__name__)

# ── CHEMINS ───────────────────────────────────────────────────────────────────
CSV_PATH     = "df_final_binaire_imputed.csv"
BASE_DIR     = "Results/transformer_clustering"

# ── MODÈLE ────────────────────────────────────────────────────────────────────
MODEL_NAME   = "yikuan8/Clinical-Longformer"
BATCH_SIZE   = 64
MAX_LENGTH   = 512

# ── HYPERPARAMÈTRES ───────────────────────────────────────────────────────────
UMAP_COMPONENTS_CLUSTERING = 10    # UMAP pour clustering
UMAP_N_NEIGHBORS           = 15
UMAP_MIN_DIST_CLUST        = 0.0
UMAP_N_NEIGHBORS_VIZ       = 30
UMAP_MIN_DIST_VIZ          = 0.1
HDBSCAN_MIN_SAMPLES        = 10
HDBSCAN_MIN_CLUSTER_SIZE   = 500   # valeur par défaut, overridé par le sweep
SWEEP_MCS_MIN              = 200
SWEEP_MCS_MAX              = 2000
SWEEP_MCS_STEP             = 100
SILHOUETTE_SAMP            = 3000

# ── COLONNES ──────────────────────────────────────────────────────────────────
cols_veinous_analysis = [
    'is_hemoglobine', 'is_leucocytes', 'is_formule_leuco', 'is_urea', 'is_creatinine',
    'is_sodium', 'is_potassium', 'is_platelets', 'is_pt', 'is_aptt',
    'is_calcium', 'is_ck', 'is_lactates', 'is_troponine', 'is_bnp',
    'is_ckmb', 'is_ddimer', 'is_crp', 'is_pct', 'is_alat',
    'is_asat', 'is_bili_total', 'is_lipase', 'is_alp', 'is_iron',
    'is_ferritin', 'is_calcium_ionized', 'is_aXa_aIIa', 'is_fibrinogen'
]
cols_imaging = [
    'ultrasound_1', 'ultrasound_2', 'ct_scan_1', 'ct_scan_2', 'ct_scan_3',
    'xray_1', 'xray_2', 'xray_3', 'mri_1', 'mri_2',
    'radio_interventional_1', 'nuclear_medicine_1'
]
cols_bio_bin = [
    'is_blood_gas', 'is_aXa_aIIa', 'is_csf', 'is_lactates', 'is_culture',
    'is_leucocytes', 'is_formule_leuco', 'is_alat', 'is_asat', 'is_bnp',
    'is_bili_total', 'is_ck', 'is_ckmb', 'is_crp', 'is_calcium_ionized',
    'is_calcium', 'is_creatinine', 'is_ddimer', 'is_iron', 'is_ferritin',
    'is_fibrinogen', 'is_hemoglobine', 'is_lipase', 'is_alp', 'is_platelets',
    'is_potassium', 'is_pct', 'is_sodium', 'is_aptt', 'is_pt',
    'is_troponine', 'is_urea', 'had_ekg'
]
cols_multi_cat = [
    'ultrasound_1', 'ultrasound_2', 'ct_scan_1', 'ct_scan_2', 'ct_scan_3',
    'xray_1', 'xray_2', 'xray_3', 'mri_1', 'mri_2',
    'radio_interventional_1', 'nuclear_medicine_1'
]
cols_quanti = ['imaging_exam_count', 'bio_exam_count']

# ── SCÉNARIOS ─────────────────────────────────────────────────────────────────
SCENARIOS = {
    "advanced_radio_bio_ekg_dispo": [
        'has_ultrasound', 'has_ct_scan', 'has_xray', 'has_mri',
        'has_radio_interventional', 'has_nuclear_medicine',
        'has_blood_test', 'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
        'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
        'imaging_exam_count', 'bio_exam_count',
    ],
    "advanced_detailed_radio_bio_ekg_dispo":
        cols_veinous_analysis + cols_imaging + [
            'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
            'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
            'imaging_exam_count', 'bio_exam_count',
        ]
}

# ── VERB MAP ──────────────────────────────────────────────────────────────────
VERB_MAP = {
    'has_ultrasound':           "underwent ultrasound",
    'has_ct_scan':              "underwent CT scan",
    'has_xray':                 "underwent X-ray",
    'has_mri':                  "underwent MRI",
    'has_radio_interventional': "underwent interventional radiology procedure",
    'has_nuclear_medicine':     "underwent nuclear medicine imaging",
    'has_blood_test':           "had blood work ordered",
    'has_culture':              "had microbiological cultures taken",
    'has_lumbar_puncture':      "underwent lumbar puncture",
    'has_blood_gas':            "had arterial blood gas analysis",
    'had_ekg':                  "had an EKG performed",
    'hospitalization':          "was admitted to the hospital",
    'observation_unit':         "was placed in observation unit",
    'inter_facility_transfer':  "was transferred to another facility",
    'is_hemoglobine':           "hemoglobin was measured",
    'is_leucocytes':            "white blood cell count was obtained",
    'is_formule_leuco':         "differential leukocyte count was performed",
    'is_urea':                  "blood urea was measured",
    'is_creatinine':            "creatinine was assessed",
    'is_sodium':                "sodium level was checked",
    'is_potassium':             "potassium level was checked",
    'is_platelets':             "platelet count was obtained",
    'is_pt':                    "prothrombin time was measured",
    'is_aptt':                  "aPTT was measured",
    'is_calcium':               "calcium level was assessed",
    'is_ck':                    "CK was measured",
    'is_lactates':              "lactate level was obtained",
    'is_troponine':             "troponin was measured",
    'is_bnp':                   "BNP was assessed",
    'is_ckmb':                  "CK-MB was measured",
    'is_ddimer':                "D-dimer was obtained",
    'is_crp':                   "CRP was measured",
    'is_pct':                   "procalcitonin was assessed",
    'is_alat':                  "ALT was measured",
    'is_asat':                  "AST was measured",
    'is_bili_total':            "total bilirubin was assessed",
    'is_lipase':                "lipase was measured",
    'is_alp':                   "ALP was measured",
    'is_iron':                  "serum iron was assessed",
    'is_ferritin':              "ferritin was measured",
    'is_calcium_ionized':       "ionized calcium was checked",
    'is_aXa_aIIa':              "anti-Xa/anti-IIa activity was measured",
    'is_fibrinogen':            "fibrinogen was assessed",
}

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
print("Loading data...")
df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"  Shape: {df.shape}")

# ── FEATURE ENGINEERING ───────────────────────────────────────────────────────
df['observation_unit']      = df['disposition'].str.contains('Obs',      case=False, na=False).astype(int)
df['inter_facility_transfer'] = df['disposition'].str.contains('transfer', case=False, na=False).astype(int)

print("\nCheck Biology (has_blood_test) :")
print(df['has_blood_test'].value_counts(normalize=True))
print("\nRépartition UHCD :")
print(df['observation_unit'].value_counts())
print("\nRépartition Transfer :")
print(df['inter_facility_transfer'].value_counts())

In [ ]:
# ============================================================
# FUNCTIONS
# ============================================================


# ── TEXT BUILDING ─────────────────────────────────────────────────────────────
def build_text(row: pd.Series, cols: list, use_natural_language: bool = True) -> str:
    parts = []
    for col in cols:
        val = row[col]
        if pd.isna(val) or val == 0 or str(val).upper() in ["NONE", "NAN", ""]:
            continue
        if isinstance(val, str):
            parts.append(val.lower())
        elif val == 1 or val == 1.0:
            if use_natural_language and col in VERB_MAP:
                parts.append(VERB_MAP[col])
            else:
                clean = col.replace('is_','').replace('has_','').replace('had_','').replace('_',' ')
                parts.append(clean)
        else:
            clean = col.replace('_', ' ')
            parts.append(f"{clean}: {val}")
    if not parts:
        return "no_resource_utilization"
    return "The patient " + ", and ".join(parts) + "."


# ── EMBEDDING ─────────────────────────────────────────────────────────────────
def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask    = attention_mask.unsqueeze(-1).float()
    summed  = (last_hidden_state * mask).sum(dim=1)
    counts  = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

@torch.no_grad()
def embed_texts(texts, tokenizer, model, device) -> np.ndarray:
    all_embeddings = []
    model.eval()
    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding"):
        batch   = texts[start: start + BATCH_SIZE]
        encoded = tokenizer(batch, padding=True, truncation=True,
                            max_length=MAX_LENGTH, return_tensors="pt").to(device)
        out     = model(**encoded)
        emb     = mean_pool(out.last_hidden_state, encoded["attention_mask"])
        all_embeddings.append(emb.cpu().float().numpy())
    return np.vstack(all_embeddings)


# ── CLUSTERING METRICS ────────────────────────────────────────────────────────
def compute_stability(emb_umap, mcs, n_runs=3, sample_frac=0.8):
    labels_runs = []
    n = len(emb_umap)
    for _ in range(n_runs):
        idx  = np.random.choice(n, int(n * sample_frac), replace=False)
        lbls = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_method="eom"
        ).fit_predict(emb_umap[idx])
        labels_runs.append((idx, lbls))
    scores = []
    for i in range(len(labels_runs)):
        for j in range(i + 1, len(labels_runs)):
            idx_i, lbl_i = labels_runs[i]
            idx_j, lbl_j = labels_runs[j]
            common = np.intersect1d(idx_i, idx_j)
            if len(common) < 10:
                continue
            li = lbl_i[np.isin(idx_i, common)]
            lj = lbl_j[np.isin(idx_j, common)]
            scores.append(adjusted_rand_score(li, lj))
    return np.mean(scores) if scores else 0.0

def compute_combined_score(df_sweep, w_sil=0.4, w_stab=0.3, w_out=0.3):
    df          = df_sweep.copy()
    df["silhouette"] = df["silhouette"].fillna(0)
    df["stability"]  = df["stability"].fillna(0)
    sil_range   = df["silhouette"].max() - df["silhouette"].min()
    sil_norm    = (df["silhouette"] - df["silhouette"].min()) / sil_range if sil_range > 0 else df["silhouette"] * 0
    stab_range  = df["stability"].max() - df["stability"].min()
    stab_norm   = (df["stability"] - df["stability"].min()) / stab_range if stab_range > 0 else df["stability"] * 0
    df["combined_score"] = w_sil * sil_norm + w_stab * stab_norm - w_out * df["outlier_rate"]
    return df


# ── VISUALISATION ─────────────────────────────────────────────────────────────
def scatter_plot(xy, labels, title, out_path):
    labels          = np.array(labels)
    unique_clusters = sorted(set(labels) - {-1})
    n_clusters      = len(unique_clusters)
    n_noise         = (labels == -1).sum()
    noise_mask      = labels == -1
    colors = plt.get_cmap("tab20")(np.linspace(0, 1, max(n_clusters, 1))) if n_clusters <= 20 \
             else plt.get_cmap("hsv")(np.linspace(0, 1, n_clusters))
    fig, ax = plt.subplots(figsize=(12, 9))
    ax.scatter(xy[noise_mask, 0], xy[noise_mask, 1], s=2, alpha=0.2, color="grey",
               label=f"Noise ({n_noise} pts)")
    for i, lbl in enumerate(unique_clusters):
        mask = labels == lbl
        ax.scatter(xy[mask, 0], xy[mask, 1], s=5, alpha=0.6, color=colors[i],
                   label=f"Cluster {lbl} (n={mask.sum()})")
    ax.set_title(f"{title}\n{n_clusters} clusters — {n_noise} noise points ({n_noise/len(labels)*100:.1f}%)")
    ax.legend(markerscale=3, fontsize=7, loc="best", ncol=max(1, n_clusters // 20))
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close(fig)

def plot_sweep_curves(df_sweep, sc_name, out_dir):
    x = df_sweep["mcs"]
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.plot(x, df_sweep["silhouette"], color="tab:blue",  marker="o", label="Silhouette")
    ax1.plot(x, df_sweep["stability"],  color="tab:red",   marker="s", label="Stabilité")
    ax1.set_xlabel("min_cluster_size")
    ax1.set_ylabel("Score (0-1)")
    ax1.legend(loc="upper left")
    ax2 = ax1.twinx()
    ax2.plot(x, df_sweep["outlier_rate"], color="tab:orange", marker="^", linestyle="--", label="Outlier rate")
    ax2.set_ylabel("Outlier rate (0-1)", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")
    ax2.legend(loc="upper right")
    plt.title(f"Sweep métriques — {sc_name}")
    fig.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{sc_name}_silhouette_stability.png"), dpi=150)
    plt.close()
    plt.figure(figsize=(10, 6))
    plt.plot(x, df_sweep["combined_score"], color="tab:green", marker="d", linewidth=2)
    plt.xlabel("min_cluster_size")
    plt.ylabel("Score combiné")
    plt.title(f"Score combiné — {sc_name}")
    plt.grid(True)
    plt.savefig(os.path.join(out_dir, f"{sc_name}_combined_score.png"), dpi=150)
    plt.close()
    log.info(f"💾 Courbes sweep sauvegardées pour {sc_name}")

def plot_wordclouds(df_valid, sc_name, out_dir):
    clusters = sorted(df_valid['cluster'].unique())
    n_cols   = 3
    n_rows   = (len(clusters) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 5))
    axes = axes.flatten()
    for i, clust in enumerate(clusters):
        texts         = df_valid[df_valid['cluster'] == clust]['text'].dropna().tolist()
        combined_text = " ".join(texts)
        wc = WordCloud(width=600, height=400, background_color="white",
                       colormap="tab10", max_words=50).generate(combined_text)
        axes[i].imshow(wc, interpolation="bilinear")
        axes[i].axis("off")
        axes[i].set_title(f"Cluster {clust} (n={len(texts)})", fontsize=12, fontweight="bold")
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.suptitle(f"Wordclouds par cluster — {sc_name}", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{sc_name}_wordclouds.png"), dpi=150)
    plt.close()
    log.info(f"💾 Wordclouds sauvegardés")

def plot_radar(df_valid, c_bio, sc_name, out_dir):
    means      = df_valid.groupby('cluster')[c_bio].mean() * 100
    N          = len(c_bio)
    angles     = [n / float(N) * 2 * np.pi for n in range(N)]
    angles    += angles[:1]
    fig, ax    = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    for clust in means.index:
        values  = means.loc[clust].tolist() + means.loc[clust].tolist()[:1]
        ax.plot(angles, values, linewidth=1.5, label=f"Cluster {clust}")
        ax.fill(angles, values, alpha=0.1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([c.replace('is_', '') for c in c_bio], size=7)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax.set_title(f"Profils radar — {sc_name}")
    plt.savefig(os.path.join(out_dir, f"{sc_name}_radar.png"), dpi=150)
    plt.close()

def plot_heatmap(df_valid, c_bio, sc_name, out_dir):
    bio_means = df_valid.groupby('cluster')[c_bio].mean() * 100
    fig, ax   = plt.subplots(figsize=(20, len(bio_means) * 0.8 + 2))
    sns.heatmap(bio_means.T, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5, ax=ax)
    ax.set_title(f"% présence par cluster — {sc_name}")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Biomarqueur")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{sc_name}_heatmap.png"), dpi=150)
    plt.close()

In [ ]:
# # ============================================================
# # PIPELINE 1 — texte → embeddings transformer → clustering initial
# # ============================================================
#
#
# def run_pipeline():
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     log.info(f"Device : {device}")
#
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     model     = AutoModel.from_pretrained(MODEL_NAME).to(device)
#
#     for sc_name, sc_cols in SCENARIOS.items():
#         sc_dir = os.path.join(BASE_DIR, sc_name)
#         os.makedirs(sc_dir, exist_ok=True)
#         log.info(f"\n🚀 SCÉNARIO : {sc_name}")
#
#         # 1. Texte
#         valid_cols  = [c for c in sc_cols if c in df.columns]
#         df['text']  = df.apply(lambda row: build_text(row, valid_cols), axis=1)
#         texts       = df['text'].tolist()
#
#         # 2. Embeddings transformer
#         embeddings  = embed_texts(texts, tokenizer, model, device)
#         np.save(os.path.join(sc_dir, f"{sc_name}_embeddings.npy"), embeddings)
#         log.info(f"💾 Embeddings sauvegardés : {sc_dir}/{sc_name}_embeddings.npy")
#
#         # 3. UMAP 10d pour clustering
#         log.info("📉 UMAP 10d...")
#         emb_umap = umap.UMAP(
#             n_components=UMAP_COMPONENTS_CLUSTERING,
#             n_neighbors=UMAP_N_NEIGHBORS,
#             min_dist=UMAP_MIN_DIST_CLUST,
#             metric="cosine",
#             random_state=42
#         ).fit_transform(embeddings)
#
#         # 4. Clustering HDBSCAN (paramètres par défaut)
#         labels = hdbscan.HDBSCAN(
#             min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
#             min_samples=HDBSCAN_MIN_SAMPLES,
#             metric="euclidean",
#             cluster_selection_method="eom"
#         ).fit_predict(emb_umap)
#
#         # 5. UMAP 2D visualisation
#         log.info("🗺️ UMAP 2D...")
#         umap_2d = umap.UMAP(
#             n_components=2,
#             n_neighbors=UMAP_N_NEIGHBORS_VIZ,
#             min_dist=UMAP_MIN_DIST_VIZ,
#             metric="cosine",
#             random_state=42
#         ).fit_transform(embeddings)
#
#         scatter_plot(umap_2d, labels, f"UMAP — {sc_name}",
#                      os.path.join(sc_dir, f"{sc_name}_umap.png"))
#
#         # 6. CSV résultats
#         res_df             = df.copy()
#         res_df['cluster']  = labels
#         res_df['umap_x']   = umap_2d[:, 0]
#         res_df['umap_y']   = umap_2d[:, 1]
#         res_df.to_csv(os.path.join(sc_dir, f"{sc_name}_results.csv"), index=False)
#         log.info(f"💾 CSV résultats sauvegardé : {sc_dir}/{sc_name}_results.csv")
#
#         # Nettoyage
#         df.drop(columns=['text'], inplace=True)
#         del embeddings, emb_umap, texts, labels, umap_2d
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#
#     log.info("\n✅ Pipeline embedding terminé !")
#
# if __name__ == "__main__":
#     run_pipeline()

In [ ]:
# ============================================================
# PIPELINE 2 — sweep HDBSCAN, score combiné, meilleure config
# ============================================================



def run_sweep():
    for sc_name in SCENARIOS:
        sc_dir  = os.path.join(BASE_DIR, sc_name)
        out_dir = os.path.join(sc_dir, "cluster_sweep")
        os.makedirs(out_dir, exist_ok=True)

        emb_path = os.path.join(sc_dir, f"{sc_name}_embeddings.npy")
        if not os.path.exists(emb_path):
            log.warning(f"❌ Embeddings introuvables : {emb_path}")
            continue

        log.info(f"\n🚀 SWEEP SCÉNARIO : {sc_name}")
        embeddings = np.load(emb_path)

        # UMAP 10d
        log.info("📉 UMAP 10d pour clustering...")
        emb_umap = umap.UMAP(
            n_components=UMAP_COMPONENTS_CLUSTERING,
            n_neighbors=UMAP_N_NEIGHBORS,
            min_dist=UMAP_MIN_DIST_CLUST,
            metric="cosine",
            random_state=42
        ).fit_transform(embeddings)

        # ── SWEEP ─────────────────────────────────────────────────────────────
        sweep_results = []
        for mcs in tqdm(range(SWEEP_MCS_MIN, SWEEP_MCS_MAX, SWEEP_MCS_STEP), desc=f"Sweep {sc_name}"):
            labels       = hdbscan.HDBSCAN(
                min_cluster_size=mcs,
                min_samples=HDBSCAN_MIN_SAMPLES,
                metric="euclidean",
                cluster_selection_method="eom"
            ).fit_predict(emb_umap)
            n_clusters   = len(set(labels)) - (1 if -1 in labels else 0)
            n_noise      = (labels == -1).sum()
            outlier_rate = n_noise / len(labels)

            if n_clusters < 3:
                log.info(f"  mcs={mcs} → {n_clusters} clusters, arrêt du sweep")
                break

            s_score = stab_score = 0.0
            if 3 <= n_clusters < 200:
                mask = labels != -1
                if np.sum(mask) > 10:
                    idx        = np.random.choice(np.where(mask)[0],
                                                  min(np.sum(mask), SILHOUETTE_SAMP), replace=False)
                    s_score    = silhouette_score(emb_umap[idx], labels[idx])
                    stab_score = compute_stability(emb_umap, mcs)

            sweep_results.append({
                "mcs": mcs, "n_clusters": n_clusters, "labels": labels,
                "silhouette": s_score, "stability": stab_score, "outlier_rate": outlier_rate,
            })
            log.info(f"  mcs={mcs:4d} | clusters={n_clusters:3d} | "
                     f"sil={s_score:.3f} | stab={stab_score:.3f} | outliers={outlier_rate*100:.1f}%")

        if not sweep_results:
            log.warning(f"⚠️ Aucun résultat pour {sc_name}")
            continue

        # Score combiné
        df_sweep = pd.DataFrame([{k: v for k, v in r.items() if k != "labels"} for r in sweep_results])
        df_sweep = compute_combined_score(df_sweep)
        for i, score in enumerate(df_sweep["combined_score"]):
            sweep_results[i]["combined_score"] = score

        # Meilleure config
        best = max(sweep_results, key=lambda x: x["combined_score"])
        log.info(f"🏆 Optimum : clusters={best['n_clusters']} | mcs={best['mcs']} | "
                 f"score={best['combined_score']:.3f} | sil={best['silhouette']:.3f} | "
                 f"stab={best['stability']:.3f} | outliers={best['outlier_rate']*100:.1f}%")

        # Sauvegarde sweep + best config
        df_sweep.to_csv(os.path.join(out_dir, f"{sc_name}_sweep_results.csv"), index=False)
        np.save(os.path.join(out_dir, f"{sc_name}_best_labels.npy"), best["labels"])
        pd.DataFrame([{k: v for k, v in best.items() if k != "labels"}])\
          .to_csv(os.path.join(out_dir, f"{sc_name}_best_config.csv"), index=False)

        plot_sweep_curves(df_sweep, sc_name, out_dir)
        log.info(f"💾 Sweep sauvegardé : {out_dir}")

    log.info("\n✅ Sweep terminé !")

if __name__ == "__main__":
    run_sweep()

In [ ]:
# # ============================================================
# # PIPELINE 3 — visualisations finales (heatmap, wordcloud, radar, UMAP/tSNE)
# # ============================================================
#
#
# def run_visualization():
#     for sc_name in SCENARIOS:
#         sc_dir  = os.path.join(BASE_DIR, sc_name)
#         out_dir = os.path.join(sc_dir, "cluster_sweep")
#
#         # Chargement embeddings + meilleurs labels
#         emb_path    = os.path.join(sc_dir,  f"{sc_name}_embeddings.npy")
#         labels_path = os.path.join(out_dir, f"{sc_name}_best_labels.npy")
#         if not os.path.exists(emb_path) or not os.path.exists(labels_path):
#             log.warning(f"❌ Fichiers manquants pour {sc_name}, skipping")
#             continue
#
#         log.info(f"\n🚀 VISUALISATION : {sc_name}")
#         embeddings = np.load(emb_path)
#         best_labels = np.load(labels_path)
#
#         # Reconstruction df_valid avec texte
#         df_sc          = df.copy()
#         df_sc["cluster"] = best_labels
#         valid_cols     = [c for c in SCENARIOS[sc_name] if c in df_sc.columns]
#         df_sc["text"]  = df_sc.apply(lambda row: build_text(row, valid_cols), axis=1)
#         df_valid       = df_sc[df_sc["cluster"] != -1].copy()
#
#         # Colonnes disponibles
#         c_bio    = [c for c in cols_bio_bin   if c in df_valid.columns]
#         c_multi  = [c for c in cols_multi_cat if c in df_valid.columns]
#         c_quanti = [c for c in cols_quanti    if c in df_valid.columns]
#
#         # Wordclouds
#         plot_wordclouds(df_valid, sc_name, out_dir)
#
#         # Radar
#         if c_bio:
#             plot_radar(df_valid, c_bio, sc_name, out_dir)
#
#         # Heatmap
#         if c_bio:
#             plot_heatmap(df_valid, c_bio, sc_name, out_dir)
#
#         # UMAP 2D + t-SNE
#         log.info("🗺️ UMAP 2D...")
#         umap_2d = umap.UMAP(
#             n_components=2, n_neighbors=UMAP_N_NEIGHBORS_VIZ,
#             min_dist=UMAP_MIN_DIST_VIZ, metric="cosine", random_state=42
#         ).fit_transform(embeddings)
#
#         log.info("🗺️ t-SNE...")
#         emb_pca50 = PCA(n_components=50, random_state=42).fit_transform(embeddings)
#         tsne_2d   = TSNE(n_components=2, perplexity=30, random_state=42,
#                          init="pca", n_jobs=-1).fit_transform(emb_pca50)
#
#         best_config_path = os.path.join(out_dir, f"{sc_name}_best_config.csv")
#         best_config      = pd.read_csv(best_config_path).iloc[0]
#
#         fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
#         title = (f"BEST MODEL : {sc_name.upper()}\n"
#                  f"Clusters: {int(best_config['n_clusters'])} | mcs: {int(best_config['mcs'])} | "
#                  f"Silhouette: {best_config['silhouette']:.3f} | "
#                  f"Stabilité: {best_config['stability']:.3f} | "
#                  f"Outliers: {best_config['outlier_rate']*100:.1f}%")
#         fig.suptitle(title, fontsize=14, fontweight="bold")
#         for ax, coord, name in [(ax1, umap_2d, "UMAP"), (ax2, tsne_2d, "t-SNE")]:
#             ax.scatter(coord[best_labels == -1, 0], coord[best_labels == -1, 1],
#                        s=1, color="lightgrey", alpha=0.2, label="Noise")
#             for l in sorted(set(best_labels) - {-1}):
#                 ax.scatter(coord[best_labels == l, 0], coord[best_labels == l, 1],
#                            s=4, alpha=0.6, label=f"C{l}")
#             ax.set_title(f"Projection {name}")
#         plt.tight_layout(rect=[0, 0.03, 1, 0.90])
#         plt.savefig(os.path.join(out_dir, f"{sc_name}_FINAL_OPTIMUM.png"), dpi=150)
#         plt.close()
#
#         # Signatures CSV
#         res_bio   = df_valid.groupby("cluster")[c_bio].mean() * 100
#         list_top3 = []
#         for col in c_multi:
#             ct = pd.crosstab(df_valid["cluster"], df_valid[col], normalize="index") * 100
#             def get_top3(row):
#                 top = row.sort_values(ascending=False).head(3)
#                 return " | ".join([f"{n} ({v:.1f}%)" for n, v in top.items() if v > 0])
#             list_top3.append(pd.DataFrame(ct.apply(get_top3, axis=1), columns=[f"Top3_{col}"]))
#
#         quanti_list = []
#         if c_quanti:
#             res_quanti_fmt = pd.DataFrame(index=df_valid.groupby("cluster")[c_quanti].mean().index)
#             for col in c_quanti:
#                 m = df_valid.groupby("cluster")[col].mean().round(1)
#                 s = df_valid.groupby("cluster")[col].std().round(1)
#                 res_quanti_fmt[f"{col}_mean±std"] = m.astype(str) + " ± " + s.astype(str)
#             quanti_list = [res_quanti_fmt]
#
#         final_sigs             = pd.concat([res_bio] + list_top3 + quanti_list, axis=1)
#         counts                 = df_valid["cluster"].value_counts()
#         final_sigs["N_patients"] = counts
#         final_sigs["Poids_%"]  = (counts / len(df) * 100).round(1)
#         final_sigs[c_bio]      = final_sigs[c_bio].round(1)
#         out_csv = os.path.join(out_dir, f"signatures_{sc_name}.csv")
#         final_sigs.sort_values("N_patients", ascending=False).to_csv(out_csv)
#         log.info(f"💾 Signatures sauvegardées : {out_csv}")
#
#     log.info("\n✅ Visualisations terminées !")
#
# if __name__ == "__main__":
#     run_visualization()